<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# User2Item recommendations with LightGCN 
We offer an example to help readers to run a ID-based collaborative filtering baseline with LightGCN. <br>
LightGCN is a simple and neat Graph Convolution Network (GCN) model for recommender systems.  I It uses a GCN to learn the embeddings of users/items, with the goal that low-order and high-order user-item interactions are explicitly exploited into the embedding function.
<img src="https://raw.githubusercontent.com/recommenders-team/resources/main/kdd2020/images%2FLightGCN-graphexample.JPG" width="600">



The model architecture is illustrated as follows:
<img src="https://raw.githubusercontent.com/recommenders-team/resources/main/images/lightGCN-model.jpg" width="600">

For more details and instructions, please refer to [lightgcn_deep_dive.ipynb](../../02_model_collaborative_filtering/lightgcn_deep_dive.ipynb).

In [1]:
import logging
import os

import pandas as pd

from recommenders.models.deeprec.DataModel.ImplicitCF import ImplicitCF
from recommenders.models.deeprec.deeprec_utils import cal_metric
from recommenders.models.deeprec.models.graphrec.lightgcn import LightGCN
from recommenders.utils.timer import Timer

from utils.general import create_dir
from utils.task_helper import group_labels, load_emb_file, prepare_dataset

logging.basicConfig(level=logging.INFO, format="%(message)s")

In [2]:
tag = "small"

In [3]:
lightgcn_dir = "data_folder/my/LightGCN-training-folder"
rawdata_dir = "data_folder/my/DKN-training-folder"
create_dir(lightgcn_dir)

First, we need to transform the raw dataset into LightGCN's input data format:

In [4]:
prepare_dataset(lightgcn_dir, rawdata_dir, tag)

load_instance_file: train_small.txt   done.
load_instance_file: valid_small.txt   done.
load_instance_file: test_small.txt   done.


In [5]:
df_train = pd.read_csv(
    os.path.join(lightgcn_dir, "lightgcn_train_{0}.txt".format(tag)),
    sep=" ",
    engine="python",
    names=["userID", "itemID", "rating"],
    header=0,
)

In [6]:
df_train.head()

,userID,itemID,rating
0,2556758139,1639559569,0
1,2556758139,2750948673,0
2,2556758139,3009232636,0
3,2556758139,1997686688,0
4,2630447844,2253252279,1


LightGCN only takes positive user-item interactions for model training. Pairs with rating < 1 will be ignored by the model.

In [7]:
df_valid = pd.read_csv(
    os.path.join(lightgcn_dir, "lightgcn_valid_{0}.txt".format(tag)),
    sep=" ",
    engine="python",
    names=["userID", "itemID", "rating"],
    header=0,
)

In [8]:
data = ImplicitCF(
    train=df_train,
    test=df_valid,
    seed=0,
    col_user="userID",
    col_item="itemID",
    col_rating="rating",
)

In [9]:
# Architecture goes to LightGCN(...); training-time params go to fit(...).
lightgcn_dir_models = os.path.join(lightgcn_dir, "saved_models")

In [10]:
model = LightGCN(
    n_users=data.n_users,
    n_items=data.n_items,
    norm_adj=data.get_norm_adj_mat(),
    embed_size=64,
    n_layers=3,
    seed=0,
)

Already create adjacency matrix.
Already normalize adjacency matrix.
Using xavier initialization.


In [11]:
with Timer() as train_time:
    model.fit(
        data,
        epochs=15,
        learning_rate=0.005,
        batch_size=1024,
        decay=0.0001,
        eval_epoch=1,
        top_k=10,
        save_model=True,
        save_epoch=1,
        model_dir=lightgcn_dir_models,
    )

print(f"Took {train_time.interval} seconds for training.")

Save model to path /home/numnum/recommenders/examples/07_tutorials/KDD2020-tutorial/data_folder/my/LightGCN-training-folder/saved_models/epoch_1
Epoch 1 (train)16.1s + (eval)1.2s: train loss = 0.07985 = (mf)0.07882 + (embed)0.00103, recall = 0.19865, ndcg = 0.10199, precision = 0.01986, map = 0.07315
Save model to path /home/numnum/recommenders/examples/07_tutorials/KDD2020-tutorial/data_folder/my/LightGCN-training-folder/saved_models/epoch_2
Epoch 2 (train)16.1s + (eval)1.0s: train loss = 0.01849 = (mf)0.01663 + (embed)0.00186, recall = 0.23405, ndcg = 0.13117, precision = 0.02341, map = 0.10009
Save model to path /home/numnum/recommenders/examples/07_tutorials/KDD2020-tutorial/data_folder/my/LightGCN-training-folder/saved_models/epoch_3
Epoch 3 (train)16.0s + (eval)1.0s: train loss = 0.01201 = (mf)0.00972 + (embed)0.00229, recall = 0.25475, ndcg = 0.14020, precision = 0.02548, map = 0.10565
Save model to path /home/numnum/recommenders/examples/07_tutorials/KDD2020-tutorial/data_folde

Took 254.99720419291407 seconds for training.


In [12]:
user_emb_file = os.path.join(lightgcn_dir, "user.emb.txt")
item_emb_file = os.path.join(lightgcn_dir, "item.emb.txt")
model.infer_embedding(user_emb_file, item_emb_file)

To compare LightGCN's performance with DKN, we need to make predictions on the same test set. So we infer the users/items embedding, then compute the similarity scores between each pairs of user-item in the test set.

In [13]:
def infer_scores_via_embeddings(test_filename, user_emb_file, item_emb_file):
    print("loading embedding file...", end=" ")
    user2vec = load_emb_file(user_emb_file)
    item2vec = load_emb_file(item_emb_file)
    preds, labels, groupids = [], [], []
    with open(test_filename, "r") as rd:
        while True:
            line = rd.readline()
            if not line:
                break
            words = line.strip().split("%")
            tokens = words[0].split(" ")
            userid = words[1]
            itemid = tokens[2]
            pred = user2vec[userid].dot(item2vec[itemid])
            preds.append(pred)
            labels.append(int(tokens[0]))
            groupids.append(userid)
    print("done")
    return labels, preds, groupids

In [14]:
test_filename = os.path.join(rawdata_dir, "test_{}.txt".format(tag))
labels, preds, group_keys = infer_scores_via_embeddings(
    test_filename, user_emb_file, item_emb_file
)
group_labels, group_preds = group_labels(labels, preds, group_keys)

loading embedding file... done


In [15]:
res_pairwise = cal_metric(group_labels, group_preds, ["ndcg@2;4;6", "group_auc"])
print(res_pairwise)
res_pointwise = cal_metric(labels, preds, ["auc"])
print(res_pointwise)

{'ndcg@2': 0.4093, 'ndcg@4': 0.5017, 'ndcg@6': 0.5395, 'group_auc': 0.8123}
{'auc': 0.813}


### Reference: 
1. Xiangnan He, Kuan Deng, Xiang Wang, Yan Li, Yongdong Zhang & Meng Wang, LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation, 2020, https://arxiv.org/abs/2002.02126